# 55 - Late Fusion TL conf60 4-class

Late Fusion (Transfer Learning): weighted average CNN + FCNN
**Dataset:** Front-only conf60 4-Class, B1 only

In [1]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import DataLoader
from collections import Counter

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from training.models import EmotionCNNTransfer, EmotionFCNN
from training.utils import (
    EmotionImageDataset, EmotionLandmarkDataset, EmotionMultimodalDataset,
    get_class_weights, train_model, full_evaluation,
    plot_training_history, plot_confusion_matrix, plot_per_class_f1
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

DATASET_DIR = PROJECT_ROOT / "data" / "dataset_frontonly_conf60_4class"
DATASET_AUG_DIR = PROJECT_ROOT / "data" / "dataset_frontonly_conf60_4class_augmented"
OUTPUT_DIR = PROJECT_ROOT / "models" / "frontonly_conf60" / "4class_tl"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 32
EPOCHS = 50
LR = 0.00005
PATIENCE = 15
NUM_CLASSES = 4
EMOTIONS = ["neutral", "happy", "sad", "negative"]

print(f"Dataset: {DATASET_DIR}")
print(f"Dataset Aug: {DATASET_AUG_DIR}")
print(f"Output: {OUTPUT_DIR}")
from sklearn.metrics import f1_score, accuracy_score

Device: cuda
GPU: Tesla T4
Dataset: /home/bs000716/MOTHER-TANK/TRAIN/data/dataset_frontonly_conf60_4class
Dataset Aug: /home/bs000716/MOTHER-TANK/TRAIN/data/dataset_frontonly_conf60_4class_augmented
Output: /home/bs000716/MOTHER-TANK/TRAIN/models/frontonly_conf60/4class_tl


In [2]:
# Load test multimodal
test_ds = EmotionMultimodalDataset(
    DATASET_DIR / "X_test_images.npy",
    DATASET_DIR / "X_test_landmarks.npy",
    DATASET_DIR / "y_test.npy")
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

CNN_DIR = PROJECT_ROOT / "models" / "frontonly_conf60" / "4class_tl"
FCNN_DIR = PROJECT_ROOT / "models" / "frontonly_conf60" / "4class"

cnn_model = EmotionCNNTransfer(num_classes=NUM_CLASSES).to(device)
fcnn_model = EmotionFCNN(num_classes=NUM_CLASSES).to(device)
cnn_model.load_state_dict(torch.load(CNN_DIR / "cnn_tl_b1.pth", map_location=device, weights_only=True))
fcnn_model.load_state_dict(torch.load(FCNN_DIR / "fcnn_b1.pth", map_location=device, weights_only=True))
cnn_model.eval(); fcnn_model.eval()

cnn_probs_all, fcnn_probs_all, labels_all = [], [], []
with torch.no_grad():
    for images, landmarks, labels in test_loader:
        cnn_probs_all.append(torch.softmax(cnn_model(images.to(device)), dim=1).cpu().numpy())
        fcnn_probs_all.append(torch.softmax(fcnn_model(landmarks.to(device)), dim=1).cpu().numpy())
        labels_all.append(labels.numpy())

cnn_probs = np.concatenate(cnn_probs_all)
fcnn_probs = np.concatenate(fcnn_probs_all)
lbls = np.concatenate(labels_all)

best_f1, best_w = 0, 0.5
for w in np.arange(0.0, 1.05, 0.05):
    preds = (w * cnn_probs + (1-w) * fcnn_probs).argmax(axis=1)
    f1 = f1_score(lbls, preds, average="macro", zero_division=0)
    if f1 > best_f1: best_f1 = f1; best_w = w; best_preds = preds

acc = accuracy_score(lbls, best_preds)
wf1 = f1_score(lbls, best_preds, average="weighted", zero_division=0)
print(f"Best CNN weight: {best_w:.2f}")
print(f"Acc={acc:.4f} Macro-F1={best_f1:.4f} W-F1={wf1:.4f}")

with open(OUTPUT_DIR / "late_fusion_tl_results.json", "w") as f:
    json.dump({"B1 Baseline": {
        "accuracy": acc, "macro_f1": best_f1, "weighted_f1": wf1, "best_cnn_weight": best_w
    }}, f, indent=2)
print(f"Saved: {OUTPUT_DIR / 'late_fusion_tl_results.json'}")

Best CNN weight: 0.60
Acc=0.8019 Macro-F1=0.5133 W-F1=0.8122
Saved: /home/bs000716/MOTHER-TANK/TRAIN/models/frontonly_conf60/4class_tl/late_fusion_tl_results.json
